<a href="https://colab.research.google.com/github/codewithUswaFatima/rag-unsloth-dynamic-4bit/blob/main/Task3_RAG_Unsloth_Dynamic4bit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3: RAG with Unsloth Dynamic 4-bit Quantization

**Pipeline:** Documents -> Chunking -> Embeddings (BGE) -> FAISS index -> Retriever -> Prompt -> Unsloth dynamic 4-bit LLM -> Grounded answer



**Why dynamic 4-bit?** Standard bitsandbytes 4-bit (NF4) quantizes *every* linear layer. Unsloth's *dynamic* quants (`*-unsloth-bnb-4bit`) analyse the quantization error per layer and keep the most sensitive layers/parameters in higher precision (16-bit), while the rest stay 4-bit. This gives noticeably better accuracy for only a small increase in VRAM.

## 1. Install dependencies

In [1]:
%%capture
!pip install unsloth
!pip install -U langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface
!pip install faiss-cpu sentence-transformers pypdf

## 2. Check the GPU

In [2]:
import unsloth
import torch, gc

assert torch.cuda.is_available(), "No GPU found. Use Runtime > Change runtime type > T4 GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} | Total VRAM: {props.total_memory/1024**3:.2f} GB")

def vram(tag=""):
    alloc = torch.cuda.memory_allocated()/1024**3
    peak  = torch.cuda.max_memory_allocated()/1024**3
    print(f"[VRAM] {tag:<35} allocated={alloc:5.2f} GB | peak={peak:5.2f} GB")

vram("baseline")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4 | Total VRAM: 14.56 GB
[VRAM] baseline                            allocated= 0.01 GB | peak= 0.01 GB


## 3. Load the Unsloth *dynamic* 4-bit model

`unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit` is a dynamic quant (note the `unsloth-bnb-4bit` suffix, which differs from the plain `bnb-4bit` upload).
Other options if you want to experiment: `unsloth/Qwen2.5-7B-Instruct-unsloth-bnb-4bit`, `unsloth/Llama-3.1-8B-Instruct-unsloth-bnb-4bit`.

In [3]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
vram("after loading dynamic 4-bit LLM")

==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


[VRAM] after loading dynamic 4-bit LLM     allocated= 2.24 GB | peak= 2.25 GB


## 4. Create the domain-specific document collection



In [4]:
import os
os.makedirs("/content/docs", exist_ok=True)

docs_text = {
"quantization_basics.txt": """
Quantization reduces the numerical precision of a neural network's weights so that the model needs less memory and can run faster.
A 7 billion parameter model stored in 16-bit precision needs roughly 14 GB just for its weights, while the same model in 4-bit precision needs
roughly 3.5 to 4 GB. This is what allows large language models to run on consumer GPUs such as the 16 GB NVIDIA T4 available in Google Colab.

NF4, or NormalFloat 4-bit, is a data type introduced with QLoRA. It is designed for weights that follow a normal distribution and is more accurate than
plain 4-bit integers for language models. Double quantization further saves memory by quantizing the quantization constants themselves.

The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates with Hugging Face Transformers through the load_in_4bit option.
Quantization-aware methods such as GPTQ and AWQ are alternatives that calibrate on sample data before quantizing.
""",
"unsloth_dynamic_quants.txt": """
Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizing every layer to 4-bit, Unsloth analyses the quantization
error of each layer and keeps the most sensitive layers and parameters in higher precision, typically 16-bit.
Models that use this approach are published with the suffix unsloth-bnb-4bit, for example Llama-3.2-3B-Instruct-unsloth-bnb-4bit.

The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badly damage some models, and in certain vision-language models
it can make the output nonsensical. Dynamic quants avoid this by not quantizing the layers that cause the largest errors.
The cost is a small increase in VRAM usage, generally under about 10 percent compared with a standard bitsandbytes 4-bit model.

Unsloth also provides FastLanguageModel, which patches the model with optimized kernels. Calling FastLanguageModel.for_inference(model) makes generation
up to about two times faster and uses less memory than the standard Hugging Face generate path.
""",
"rag_overview.txt": """
Retrieval-Augmented Generation, or RAG, combines a retriever with a generator. Documents are split into chunks, each chunk is converted into a vector
embedding, and the embeddings are stored in a vector database such as FAISS. At query time the question is embedded, the most similar chunks are
retrieved, and they are inserted into the prompt of the language model as context.

RAG reduces hallucination because the model is instructed to answer only from the retrieved context. It also lets a model use private or recent
knowledge without retraining. Important design choices are the chunk size, the chunk overlap, the embedding model, and the number of retrieved chunks, often called top-k.
Small chunks give precise matches but lose context, while large chunks keep context but add noise and use more of the context window and VRAM.

LoRA and QLoRA are fine-tuning methods that train small adapter matrices on top of a frozen, quantized base model. They are often combined with RAG:
RAG supplies facts, while fine-tuning teaches style and task format.
""",
}

for name, text in docs_text.items():
    with open(f"/content/docs/{name}", "w") as f:
        f.write(text.strip())
print(os.listdir("/content/docs"))

['rag_overview.txt', 'unsloth_dynamic_quants.txt', 'quantization_basics.txt']


## 5. Load documents and split into chunks

In [5]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob

def load_documents(folder="/content/docs"):
    documents = []
    for path in sorted(glob.glob(f"{folder}/*")):
        if path.lower().endswith(".pdf"):
            documents.extend(PyPDFLoader(path).load())
        elif path.lower().endswith((".txt", ".md")):
            with open(path, encoding="utf-8", errors="ignore") as f:
                documents.append(Document(page_content=f.read(), metadata={"source": os.path.basename(path)}))
    return documents

documents = load_documents()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80,
                                          separators=["\n\n", "\n", ". ", " "])
chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents -> {len(chunks)} chunks")
print("\nSample chunk:\n", chunks[0].page_content[:300], "\n", chunks[0].metadata)

/tmp/ipykernel_465/2721922506.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


3 documents -> 9 chunks

Sample chunk:
 Quantization reduces the numerical precision of a neural network's weights so that the model needs less memory and can run faster.
A 7 billion parameter model stored in 16-bit precision needs roughly 14 GB just for its weights, while the same model in 4-bit precision needs
roughly 3.5 to 4 GB. This  
 {'source': 'quantization_basics.txt'}


## 6. Embeddings + FAISS vector index

`BAAI/bge-small-en-v1.5` is a small (~130 MB) embedding model that uses only a few hundred MB of VRAM, so almost all GPU memory is left for the LLM.

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("/content/faiss_index")      # persist so you can reload later
vram("after building FAISS index")

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[VRAM] after building FAISS index          allocated= 2.36 GB | peak= 2.38 GB


### Retrieval test (with similarity scores)

In [7]:
query = "What is special about Unsloth dynamic 4-bit quantization?"
for doc, score in vectorstore.similarity_search_with_score(query, k=3):
    print(f"score={score:.3f} | {doc.metadata.get('source')}\n  {doc.page_content[:200]}...\n")

score=0.192 | unsloth_dynamic_quants.txt
  Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizing every layer to 4-bit, Unsloth analyses the quantization
error of each layer and keeps the most sensitive la...

score=0.368 | quantization_basics.txt
  The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates with Hugging Face Transformers through the load_in_4bit option.
Quantization-aware methods such as GPTQ and AW...

score=0.414 | unsloth_dynamic_quants.txt
  The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badly damage some models, and in certain vision-language models
it can make the output nonsensical. Dynamic quants...



## 7. Build the RAG chain (LangChain LCEL + Unsloth LLM)

The chain is: `question -> retriever -> format context -> chat prompt -> quantized LLM -> answer`.
The system prompt forces the model to answer **only** from the context (grounded generation).

In [8]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

SYSTEM_PROMPT = (
    "You are a precise assistant. Answer the question using ONLY the provided context. "
    "If the context does not contain the answer, reply exactly: 'I could not find this in the provided documents.' "
    "Keep the answer concise."
)

def format_docs(docs):
    return "\n\n".join(f"[Source {i+1}: {d.metadata.get('source','?')}]\n{d.page_content}" for i, d in enumerate(docs))

@torch.inference_mode()
def generate_answer(inputs: dict, max_new_tokens=300, temperature=0.2) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{inputs['context']}\n\nQuestion: {inputs['question']}"},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to("cuda")
    out = model.generate(
        **prompt_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True, temperature=temperature, top_p=0.9,
        use_cache=True, pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][prompt_ids["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | RunnableLambda(generate_answer)
)

def ask(question, show_sources=True):
    print(f"Q: {question}")
    if show_sources:
        for d in retriever.invoke(question):
            print(f"   - retrieved from {d.metadata.get('source')}: {d.page_content[:90].strip()}...")
    print(f"A: {rag_chain.invoke(question)}\n" + "-"*80)

## 8. Demo: grounded answers

In [9]:
torch.cuda.reset_peak_memory_stats()

ask("What does Unsloth's dynamic 4-bit quantization do differently from standard 4-bit?")
ask("How much VRAM extra does dynamic quantization need?")
ask("How much memory does a 7B model need in 4-bit versus 16-bit?")
ask("Why does RAG reduce hallucination?")
ask("What is NF4 and where was it introduced?")

vram("after RAG demo queries")

Q: What does Unsloth's dynamic 4-bit quantization do differently from standard 4-bit?
   - retrieved from unsloth_dynamic_quants.txt: Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizi...
   - retrieved from unsloth_dynamic_quants.txt: The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badl...
   - retrieved from quantization_basics.txt: The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates...
   - retrieved from quantization_basics.txt: NF4, or NormalFloat 4-bit, is a data type introduced with QLoRA. It is designed for weight...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Unsloth's dynamic 4-bit quantization avoids quantizing the layers that cause the largest errors, unlike standard 4-bit quantization which quantizes every layer to 4-bit.
--------------------------------------------------------------------------------
Q: How much VRAM extra does dynamic quantization need?
   - retrieved from unsloth_dynamic_quants.txt: The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badl...
   - retrieved from quantization_basics.txt: Quantization reduces the numerical precision of a neural network's weights so that the mod...
   - retrieved from quantization_basics.txt: The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates...
   - retrieved from unsloth_dynamic_quants.txt: Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizi...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: About 10 percent compared with a standard 4-bit model.
--------------------------------------------------------------------------------
Q: How much memory does a 7B model need in 4-bit versus 16-bit?
   - retrieved from quantization_basics.txt: Quantization reduces the numerical precision of a neural network's weights so that the mod...
   - retrieved from quantization_basics.txt: The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates...
   - retrieved from unsloth_dynamic_quants.txt: The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badl...
   - retrieved from quantization_basics.txt: NF4, or NormalFloat 4-bit, is a data type introduced with QLoRA. It is designed for weight...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: According to Source 1, a 7 billion parameter model stored in 16-bit precision needs roughly 14 GB, while the same model in 4-bit precision needs roughly 3.5 to 4 GB.
--------------------------------------------------------------------------------
Q: Why does RAG reduce hallucination?
   - retrieved from rag_overview.txt: RAG reduces hallucination because the model is instructed to answer only from the retrieve...
   - retrieved from rag_overview.txt: Retrieval-Augmented Generation, or RAG, combines a retriever with a generator. Documents a...
   - retrieved from unsloth_dynamic_quants.txt: The main benefit of dynamic quantization is accuracy. Standard 4-bit quantization can badl...
   - retrieved from rag_overview.txt: LoRA and QLoRA are fine-tuning methods that train small adapter matrices on top of a froze...


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: RAG reduces hallucination because the model is instructed to answer only from the retrieved context.
--------------------------------------------------------------------------------
Q: What is NF4 and where was it introduced?
   - retrieved from quantization_basics.txt: NF4, or NormalFloat 4-bit, is a data type introduced with QLoRA. It is designed for weight...
   - retrieved from unsloth_dynamic_quants.txt: Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizi...
   - retrieved from quantization_basics.txt: The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates...
   - retrieved from quantization_basics.txt: Quantization reduces the numerical precision of a neural network's weights so that the mod...
A: NF4, or NormalFloat 4-bit, is a data type. It was introduced with QLoRA.
--------------------------------------------------------------------------------
[VRAM] after RAG demo queries              allocated= 2.

### Out-of-scope question (the model should refuse instead of hallucinating)

In [10]:
ask("Who won the FIFA World Cup in 2018?")

Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Who won the FIFA World Cup in 2018?
   - retrieved from quantization_basics.txt: NF4, or NormalFloat 4-bit, is a data type introduced with QLoRA. It is designed for weight...
   - retrieved from quantization_basics.txt: The bitsandbytes library provides 4-bit and 8-bit quantization for PyTorch and integrates...
   - retrieved from unsloth_dynamic_quants.txt: Unsloth dynamic 4-bit quantization is a selective quantization method. Instead of quantizi...
   - retrieved from quantization_basics.txt: Quantization reduces the numerical precision of a neural network's weights so that the mod...
A: I could not find this in the provided documents.
--------------------------------------------------------------------------------


### With RAG vs. without RAG

In [11]:
@torch.inference_mode()
def plain_answer(question, max_new_tokens=200):
    msgs = [{"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

q = "In the provided documents, which suffix marks Unsloth dynamic 4-bit models and how much extra VRAM do they use?"
print("WITHOUT RAG:\n", plain_answer(q), "\n")
print("WITH RAG:\n", rag_chain.invoke(q))

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


WITHOUT RAG:
 I'm not able to find any information about the provided documents. However, I can tell you that Unsloth dynamic 4-bit models typically use the "U" suffix. 

Unfortunately, I do not have information on how much extra VRAM they use. 

WITH RAG:
 The suffix that marks Unsloth dynamic 4-bit models is -unsloth-bnb-4bit. According to Source 3, these models use approximately 10% more VRAM than standard 4-bit models.


## 9. Memory report



In [12]:
gc.collect(); torch.cuda.empty_cache()
vram("final")
total = props.total_memory/1024**3
peak = torch.cuda.max_memory_allocated()/1024**3
print(f"Peak usage = {peak:.2f} GB of {total:.1f} GB ({100*peak/total:.1f}% of GPU)")
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

[VRAM] final                               allocated= 2.51 GB | peak= 2.58 GB
Peak usage = 2.58 GB of 14.6 GB (17.7% of GPU)
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 2745 MiB, 15360 MiB


## 10. Used my own documents
Upload PDFs/TXT files, rebuild the index and ask questions.

In [13]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    open(f"/content/docs/{name}", "wb").write(data)

documents = load_documents()
chunks = splitter.split_documents(documents)
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
rag_chain = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | RunnableLambda(generate_answer)
)
print(f"Re-indexed {len(documents)} docs / {len(chunks)} chunks")
ask("Summarize the main idea of the uploaded document.")

Saving AI401_Lecture01_Introduction_to_KRR (2).pdf to AI401_Lecture01_Introduction_to_KRR (2).pdf


Both `max_new_tokens` (=300) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Re-indexed 26 docs / 41 chunks
Q: Summarize the main idea of the uploaded document.
   - retrieved from /content/docs/AI401_Lecture01_Introduction_to_KRR (2).pdf: information and action.
Core idea
Knowledge + reasoning → informed action...
   - retrieved from /content/docs/AI401_Lecture01_Introduction_to_KRR (2).pdf: represent useful knowledge so a machine can reason with it?”...
   - retrieved from rag_overview.txt: Retrieval-Augmented Generation, or RAG, combines a retriever with a generator. Documents a...
   - retrieved from /content/docs/AI401_Lecture01_Introduction_to_KRR (2).pdf: KRR question
What should be represented, and how shou ld 
the agent reason with it?...
A: The main idea of the uploaded document is that Knowledge + reasoning → informed action.
--------------------------------------------------------------------------------
